# Generate 1st/last frame

In [ ]:
# Imports.
import dv_processing as dv  # dv_processing: event-camera I/O.
import numpy as np  # NumPy.
import cv2 as cv
import os
import glob
from tqdm import tqdm

def read_frame(file_path):
    """
    Read the first and last frames from an AEDAT4 file.
    Returns (first_image, last_image, frame_count) or (None, None) on failure.
    """
    recording = dv.io.MonoCameraRecording(file_path)

    if not recording.isFrameStreamAvailable():
        return None, None

    frame = recording.getNextFrame()
    first_image = frame.image
    frame_count = 1
    while frame is not None:
        last_image = frame.image
        frame = recording.getNextFrame()
        frame_count += 1

    return first_image, last_image, frame_count

def batch_process_aedat4_files(input_folder, output_folder):
    """
    Batch-process every AEDAT4 file in a folder.
    
    Args:
        input_folder: directory containing AEDAT4 files.
        output_folder: directory for the output PNG files.
    """
    
    # Ensure the output folder exists.
    os.makedirs(output_folder, exist_ok=True)
    
    # Enumerate the AEDAT4 files.
    aedat4_pattern = os.path.join(input_folder, "*.aedat4")
    aedat4_files = glob.glob(aedat4_pattern)  
    print(f"Found {len(aedat4_files)} AEDAT4 files.")
    
    success_count = 0
    error_count = 0
    failed_files = []
    
    # Progress bar.
    for file_path in tqdm(aedat4_files, desc="Processing AEDAT4", unit="file"):
        try:
            # Derive the basename without extension.
            file_name = os.path.splitext(os.path.basename(file_path))[0]
            output_path = os.path.join(output_folder, f"{file_name}_frame.png")

            # Read first and last frames.
            first_img, last_img, frame_count = read_frame(file_path)
            if first_img is None or last_img is None:
                raise ValueError("frame stream missing or empty")

            # combined = np.hstack([first_img, last_img])
            # cv.imwrite(output_path, combined)
            cv.imwrite(output_path, last_img)
            print(f"file_name: {file_name}, frame_count: {frame_count}")
            success_count += 1
                
        except Exception as e:
            error_count += 1
            failed_files.append((file_path, str(e)))
    
    print("\nBatch processing complete.")
    print(f"Succeeded: {success_count} files")
    print(f"Failed:    {error_count} files")
    if failed_files:
        print("Failed file list:")
        for file_path, reason in failed_files:
            print(f"- {file_path} : {reason}")

if __name__ == "__main__":
    # Configuration.
    # input_folder = "cifar10_xdvs_preview"  # directory containing AEDAT4 files
    input_folder = "./data"  # directory containing AEDAT4 files
    output_folder = "aedat_preview_lastframe"  # output PNG directory
    
    # Run batch processing.
    batch_process_aedat4_files(input_folder, output_folder)


In [ ]:
# DCT-based pHash + Hamming distance + union-find duplicate grouping (memory-optimized).  
import cv2  # Image processing.
import numpy as np  # Numerics.
import os  # Path utilities.
import glob  # Glob matching.
import sys  # sys.path.
import gc  # Garbage collection.
from tag_detector import process_frame, create_detector  # Import the crop pipeline and detector factory.

# ---------- pHash (DCT-based perceptual hash) and helpers ----------

def calculate_phash_dct(image, hash_size=8, highfreq_factor=4):  # Compute the DCT-based perceptual hash.
    # image: BGR or grayscale input. hash_size: low-frequency block edge. highfreq_factor: upsampling factor.
    if len(image.shape) == 3:  # Color input.
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale.
    else:
        gray = image  # Already grayscale.
    size = hash_size * highfreq_factor  # e.g. 8*4 = 32.
    resized = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA)  # Resize to the high-frequency grid.
    resized = np.float32(resized)  # Cast to float32 for DCT.
    dct = cv2.dct(resized)  # 2D DCT.
    lowfreq = dct[:hash_size, :hash_size]  # Top-left low-frequency block (8x8).
    lowfreq_flat = lowfreq.flatten()  # Flatten.
    median_val = np.median(lowfreq_flat[1:])  # Median excluding the DC term.
    bits = lowfreq_flat > median_val  # Threshold against the median to obtain bits.
    hash_value = 0  # Initialize a 64-bit integer.
    for i, bit in enumerate(bits):  # Iterate over the 64 bits.
        if bit:  # If set.
            hash_value |= 1 << i  # Set that bit.
    return int(hash_value)  # Return as an int hash.

def hamming_distance(hash1, hash2):  # Hamming distance using the fast bit_count implementation.
    return (hash1 ^ hash2).bit_count()  # bit_count is available natively in Python 3.10+.

def is_black_or_white_frame(image, black_threshold=0.01, white_threshold=0.95):  # Detect a near-empty (all-black or all-white) frame.
    # black_threshold: mean threshold for near-black. white_threshold: mean threshold for near-white.
    if len(image.shape) == 3:  # Color.
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # To grayscale.
    else:
        gray = image  # Already grayscale.
    norm = gray.astype(np.float32) / 255.0  # Normalize to [0, 1].
    mean_val = float(np.mean(norm))  # Mean.
    std_val = float(np.std(norm))  # Std deviation.
    is_black = (mean_val < black_threshold) and (std_val < 0.05)  # Near-black test.
    is_white = (mean_val > white_threshold) and (std_val < 0.05)  # Near-white test.
    return is_black, is_white, mean_val, std_val  # Return flags and stats.

# ---------- Union-find for duplicate-group merging ----------

class UnionFind:  # Union-find.
    def __init__(self, n):  # Initialize.
        self.parent = list(range(n))  # Parent pointers.
        self.rank = [0] * n  # Tree rank.
    def find(self, x):  # Find with path compression.
        if self.parent[x] != x:  # Not yet a root.
            self.parent[x] = self.find(self.parent[x])  # Recursively compress.
        return self.parent[x]  # Return the root.
    def union(self, x, y):  # Union by rank.
        rx, ry = self.find(x), self.find(y)  # Find roots.
        if rx == ry:  # Already in the same set.
            return  # Nothing to do.
        if self.rank[rx] < self.rank[ry]:  # Merge by rank.
            self.parent[rx] = ry  # rx becomes a child of ry.
        elif self.rank[rx] > self.rank[ry]:  # Other direction.
            self.parent[ry] = rx  # ry becomes a child of rx.
        else:
            self.parent[ry] = rx  # Arbitrary merge.
            self.rank[rx] += 1  # Bump rank.

# ---------- Main: compute pHash, detect empty frames, locality-bucketed comparison, union-find ----------

def detect_phash_duplicates_and_empty_frames(folder_path,  # Entry point.
                                             similarity_threshold=10,  # Hamming-distance threshold (smaller = stricter).
                                             window_size=16,  # Sorted-neighborhood window size.
                                             hash_size=8,  # pHash low-frequency block size (8x8).
                                             highfreq_factor=4,  # Upsampling factor (-> 32x32 grid).
                                             gc_interval=50  # Garbage collection interval.
                                             ):
    print(f"Scanning folder: {folder_path}")
    detector = create_detector()  # AprilTag detector.
    image_files = glob.glob(os.path.join(folder_path, "*.png"))  # Collect PNG files.
    if not image_files:  # Nothing to do.
        print("No PNG files found.")
        return None  # Return empty.
    print(f"Found {len(image_files)} images.")

    records = []  # Per-image records.
    empty_frames = []  # Empty-frame log.
    failed_count = 0  # Failure counter.

    print("Computing crops and pHash...")
    for idx, image_path in enumerate(image_files):  # Iterate.
        filename = os.path.basename(image_path)  # Extract filename.
        try:
            img = cv2.imread(image_path)  # Read image.
            if img is None:  # Read failed.
                failed_count += 1  # Count.
                continue  # Skip.
            barbara_info, cropped, _ = process_frame(  # Run the crop pipeline.
                img, 0, detector,
                margin_ratio=0.03,
                tag_ref_width=287,
                barbara_ref_size=861,
                barbara_gap=82,
                is_raw=False
            )
            if cropped is None:  # No crop region detected.
                failed_count += 1  # Count.
                continue  # Skip.
            is_black, is_white, mean_val, std_val = is_black_or_white_frame(cropped)  # Test for empty frame.
            if is_black or is_white:  # Empty frame.
                empty_frames.append({  # Record.
                    'filename': filename,
                    'type': 'all-black' if is_black else 'all-white',
                    'mean': mean_val,
                    'std': std_val
                })
            ph = calculate_phash_dct(cropped, hash_size=hash_size, highfreq_factor=highfreq_factor)  # Compute pHash.
            records.append({  # Save record.
                'index': len(records),  # Index.
                'filename': filename,  # Filename.
                'path': image_path,  # Path.
                'phash': ph  # pHash value.
            })
        except Exception as e:  # Catch any exception.
            failed_count += 1  # Count.
            continue  # Skip.
        finally:
            if (idx + 1) % gc_interval == 0:  # Periodic cleanup.
                gc.collect()  # Garbage collection.
        if (idx + 1) % 100 == 0:  # Progress.
            print(f"Processed: {idx + 1}/{len(image_files)}")

    if not records:  # No valid records.
        print("No valid cropped images available for comparison.")
        return None  # Return.

    print("Bucketing by hash prefix and comparing within each bucket...")
    # 1) Group identical hashes — same hash means identical pHash, distance 0.
    exact_map = {}  # exact_hash -> list of indices.
    for rec in records:  # Iterate records.
        exact_map.setdefault(rec['phash'], []).append(rec['index'])  # Append to bucket.

    # 2) Bucket by hash prefix; only compare within the same bucket.
    # Note: because pHash is packed LSB-first, the 'prefix' is the low k bits, not the high ones.
    k = 12  # Prefix length in bits (12-16 is typical; larger -> smaller buckets, fewer comparisons).
    prefix_mask = (1 << k) - 1  # Take the low k bits as the prefix (LSB-first).
    buckets = {}  # prefix -> [(hash, index)].
    for rec in records:  # Iterate.
        h = rec['phash']  # Take the hash.
        prefix = h & prefix_mask  # Low k bits as the prefix.
        buckets.setdefault(prefix, []).append((h, rec['index']))  # Drop into the bucket.

    uf = UnionFind(len(records))  # Initialize union-find.

    # Merge images with identical hashes (distance = 0) first.
    for indices in exact_map.values():  # Iterate over each hash bucket.
        root = indices[0]  # Use the first index as a representative.
        for j in indices[1:]:  # Merge the rest.
            uf.union(root, j)  # Union.

    # Pairwise comparison inside each bucket (could be sped up with sorted neighborhoods or a BK-tree).
    for prefix, items in buckets.items():  # Iterate over each bucket.
        n = len(items)  # Bucket size.
        # Brute-force pairwise comparison (fine when n is small; consider a local window otherwise).
        for i in range(n):  # Outer.
            h1, idx1 = items[i]  # Take one item.
            for j in range(i + 1, n):  # Compare against subsequent items.
                h2, idx2 = items[j]  # Take the other item.
                dist = hamming_distance(h1, h2)  # Hamming distance.
                if dist <= similarity_threshold:  # Within threshold.
                    uf.union(idx1, idx2)  # Merge into the same group.

    # Collect the union-find connected components as duplicate groups.
    groups = {}  # root -> member filenames.
    for rec in records:  # Iterate records.
        root = uf.find(rec['index'])  # Find root.
        groups.setdefault(root, []).append(rec['filename'])  # Append filename.

    duplicate_groups = [members for members in groups.values() if len(members) >= 2]  # Keep only the groups with at least two members.

    # Print the results.
    print("\n" + "=" * 60)  # Separator.
    print("Analysis summary:")
    print(f"Total images: {len(image_files)}")
    print(f"Valid crops: {len(records)}")
    print(f"Failed:      {failed_count}")
    print(f"Duplicate groups: {len(duplicate_groups)}")
    print(f"Empty frames:     {len(empty_frames)}")
    print("=" * 60)  # Separator.

    if duplicate_groups:  # If any duplicate groups exist.
        print("\nDuplicate-group details:")
        for gi, members in enumerate(sorted(duplicate_groups, key=len, reverse=True), 1):  # Largest groups first.
            print(f"Group {gi} (size {len(members)}): {members}")

    if empty_frames:  # If any empty frames exist.
        print("\nEmpty-frame details:")
        for item in empty_frames:  # Iterate.
            print(f"{item['filename']}: {item['type']} (mean {item['mean']:.3f}, std {item['std']:.3f})")

    # Return the structured result.
    return {
        'duplicate_groups': duplicate_groups,  # duplicate groups.
        'empty_frames': empty_frames,  # empty frames.
        'valid_count': len(records),  # valid count.
        'failed_count': failed_count,  # failure count.
        'total_count': len(image_files)  # total count.
    }

# ---------- Run ----------
folder_path = "aedat_preview_lastframe"  # Image folder.
results = detect_phash_duplicates_and_empty_frames(  # Run the main pipeline.
    folder_path=folder_path,  # Path.
    similarity_threshold=10,  # Hamming-distance threshold (8-12 typical).
    window_size=16,  # Sorted-neighborhood window.
    hash_size=8,  # pHash size.
    highfreq_factor=4,  # High-frequency upsampling factor.
    gc_interval=50  # Garbage collection interval.
)
print("\nDetection complete.")